In [1]:
#!/usr/bin/env python3
import os
import pandas as pd

def reverse_readline(fh, buf_size=8192):
    """
    A generator that returns the lines of a file in reverse order.
    Reads file by blocks from the end. (Works for text files opened in binary mode.)
    """
    segment = None
    offset = 0
    fh.seek(0, os.SEEK_END)
    position = fh.tell() # position where the file was read
    file_size = fh.tell()
    while offset < file_size:
        # Incrementally decrease the position of read
        offset = min(file_size, offset + buf_size)
        position = file_size - offset
        fh.seek(position)

        # Read a buffer size of data
        buffer = fh.read(min(buf_size, offset))

        # split the buffer by newline
        lines = buffer.split(b'\n')

        # The first segment of the current buffer is likely a partial line, so add it to the previous segment.
        if segment is not None:
            if buffer[-1] != ord(b'\n'):
                # If the last char isn't a newline, then the last element is partial.
                lines[-1] += segment
            else:
                lines.append(segment)
        segment = lines[0]

        for line in reversed(lines[1:]):
            line = line.decode('utf-8', errors='replace')
            yield line.strip(), position
    # yield the last remaining segment
    if segment is not None:
        segment = segment.decode('utf-8', errors='replace')
        yield segment.strip(), position

def forward_read_lines(header, date_time, log_file_path, start_offset, num_lines=50):
    """
    Open the file in forward (normal) mode, seek to start_offset, and read num_lines.
    Returns a list of strings (lines).
    """
    lines = []
    with open(log_file_path, "r", encoding="utf-8", errors="replace") as f:
        f.seek(start_offset - 81920) # make sure enough lines are read

        marker_found = False
        count = 0
        while (count < num_lines):
            line = f.readline()

            if header in line and date_time in line:
                marker_found = True
                continue

            if marker_found:
                lines.append(line.rstrip("\n"))
                count = count + 1
    return lines


def parse_table_line(line, header):
    """
    Given a table line starting with <i> <j> <k> ..., parse it into tokens.
    """
    tokens = line.split()

    if header == "Post-reaction cec cation":
        index = ['c','j','icat','cec_cation_vr', 'meq cec_cation_vr', 
                 '-cec_cation_flux_vr * dt', '-cec_cation_flux2_vr * dt', 
                 'background_cec_vr * dt']
    elif header == "Post-reaction cation":
        index = ['c','j','icat','cation_vr','mol cation_vr',
                 'background_flux_vr * dt', 'primary_cation_flux_vr * dt', 
                 'cec_cation_flux_vr * dt', 'cec_cation_flux2_vr * dt', 
                 '-secondary_cation_flux_vr * dt', 
                 '-cation_uptake_vr * dt', 'cation_infl_vr * dt', 
                 '-cation_leached_vr * dt', 'cation_runoff_vr * dt']

    tokens = pd.Series(tokens, index = index).astype(float)

    # You may want to convert tokens to appropriate types, e.g. int or float.
    # For now, we simply return the tokens.
    return tokens

def main():
    # Change this to your error log file path
    log_file_path = "/gpfs/wolf2/cades/cli185/proj-shared/zdr/ERW/output/UQ/pft1/20250403_conus_ICB20TRCNPRDCTCBC_erw_pft1_ens0055/run/fort.202"

    # Set the problematic grid cell and time step
    latitude = f'{44.75:.15f}'
    longitude = f'{360 - 290.75:.15f}'
    date_time = '2064-06-22_21:00:00'

    # Open the file in binary mode for reverse reading for actual info. 
    collected_lines = {"Post-reaction cec cation": [],
                       "Post-reaction cation": []}  # Will collect lines from diagnostics upward to the key marker
    with open(log_file_path, "rb") as fh:
        # Read lines in reverse
        for line, position in reverse_readline(fh):
            filt = latitude in line and longitude in line and date_time in line
            if filt:
                if "Post-reaction cec cation" in line:
                    collected_lines["Post-reaction cec cation"] = forward_read_lines("Post-reaction cec cation", date_time, log_file_path, position)
                    continue
                if "Post-reaction cation" in line:
                    collected_lines["Post-reaction cation"] = forward_read_lines("Post-reaction cation", date_time, log_file_path, position)
                    break

    # Reverse collected_lines so that they are in original order (from "Post-reaction cec cation" down to diagnostics)
    for key in collected_lines.keys():
        table_lines = collected_lines[key]

        table_lines.reverse()

        for i, line in enumerate(table_lines):
            stripped = line.lstrip()
            if stripped and stripped[0].isdigit():
                table_lines[i] = parse_table_line(line, key)
            else:
                print("Table line not found after 'Post-reaction cec cation' marker.")
                return

        table_lines = pd.DataFrame(table_lines)
        table_lines['c'] = table_lines['c'].astype(int)
        table_lines['j'] = table_lines['j'].astype(int)
        table_lines['icat'] = table_lines['icat'].astype(int)
        table_lines = table_lines.set_index(['c','j','icat']).sort_index()

        collected_lines[key] = table_lines
    
    return collected_lines


if __name__ == '__main__':
    collected_lines = main()


In [5]:
collected_lines['Post-reaction cation']

cation_vr  mol cation_vr  background_flux_vr * dt  \
c     j  icat                                                         
20239 1  1     1.731286e-02   5.072798e-06             0.000000e+00   
         2     1.601562e-01   7.738076e-05             0.000000e+00   
         3     6.199503e-03   3.166670e-06             0.000000e+00   
         4     1.202690e-02   3.612274e-06             0.000000e+00   
         5     6.085846e-07   2.648890e-10             1.597081e-09   
      2  1     8.297123e-02   1.459460e-05             0.000000e+00   
         2     3.544976e-01   1.028225e-04             0.000000e+00   
         3     1.241102e-02   3.805740e-06             0.000000e+00   
         4     2.774144e-02   5.001978e-06             0.000000e+00   
         5     1.055589e-06   2.758187e-10             0.000000e+00   
      3  1     6.037002e-02   1.007777e-05             0.000000e+00   
         2     5.334990e-01   1.468545e-04             0.000000e+00   
         3     1.749813e-02   5.092159e-06             0.000000e+00   
         4     3.645022e-02   6.237227e-06             0.000000e+00   
         5     5.147783e-08   1.276520e-11             0.000000e+00   
      4  1     1.125007e-01   1.807041e-05             0.000000e+00   
         2     8.076898e-01   2.139280e-04             0.000000e+00   
         3     2.527644e-02   7.077755e-06             0.000000e+00   
         4     5.209856e-02   8.578007e-06             0.000000e+00   
         5     1.171119e-06   2.794329e-10             0.000000e+00   
      5  1     1.625873e-01   2.313901e-05             0.000000e+00   
         2     1.398834e+00   3.282729e-04             0.000000e+00   
         3     4.881151e-02   1.211010e-05             0.000000e+00   
         4     4.386846e-02   6.399689e-06             0.000000e+00   
         5     2.140643e-07   4.525502e-11             0.000000e+00   
      6  1     1.021922e+00   1.182446e-04             0.000000e+00   
         2     1.508757e+00   2.878681e-04             0.000000e+00   
         3     1.738133e-01   3.506016e-05             0.000000e+00   
         4     1.830946e-01   2.171638e-05             0.000000e+00   
         5     7.547525e-04   1.297276e-07             2.259439e-10   
      7  1     3.378301e-01   3.476208e-05             0.000000e+00   
         2     2.346241e-01   3.980983e-05             0.000000e+00   
         3     1.232503e-01   2.210866e-05             0.000000e+00   
         4     6.208233e-02   6.548224e-06             0.000000e+00   
         5     2.793551e-05   4.270002e-09             0.000000e+00   
      8  1     2.161384e-01   2.034677e-05             0.000000e+00   
         2     1.324038e-01   2.055296e-05             0.000000e+00   
         3     8.093379e-02   1.328191e-05             0.000000e+00   
         4     5.167976e-02   4.986918e-06             0.000000e+00   
         5     2.037461e-05   2.849161e-09             0.000000e+00   
      9  1     1.326857e-01   1.288090e-05             0.000000e+00   
         2     7.283559e-02   1.165940e-05             0.000000e+00   
         3     4.814655e-02   8.148061e-06             0.000000e+00   
         4     3.693879e-02   3.675808e-06             0.000000e+00   
         5     4.666773e-06   6.729810e-10             0.000000e+00   
      10 1     1.267067e-01   1.380816e-05             0.000000e+00   
         2     6.870021e-02   1.234539e-05             0.000000e+00   
         3     4.272484e-02   8.116781e-06             0.000000e+00   
         4     3.398079e-02   3.795927e-06             0.000000e+00   
         5     4.658097e-06   7.540652e-10             5.624488e-15   

               primary_cation_flux_vr * dt  cec_cation_flux_vr * dt  \
c     j  icat                                                         
20239 1  1                    0.000000e+00            -1.340367e-03   
         2                    1.870883e-43            -5.280120e-03   
         3                    0.000000e+0

In [6]:
collected_lines['Post-reaction cec cation']

cec_cation_vr  meq cec_cation_vr  -cec_cation_flux_vr * dt  \
c     j  icat                                                               
20235 1  1        253.512032           0.793647              1.752529e-02   
         2        183.412086           0.946819              7.434089e-02   
         3          6.772914           0.018482              5.393295e-03   
         4          9.398937           0.015081              9.905626e-04   
         5          5.851120           0.040815             -5.291306e-05   
      2  1        538.293292           1.685185              3.081348e-03   
         2       1618.100509           8.353038              4.374328e-02   
         3         22.301497           0.060855              2.481993e-03   
         4         21.254420           0.034103              2.527117e-04   
         5         78.139683           0.545074             -3.566971e-05   
      3  1        158.310395           0.496668              4.569233e-04   
         2       4022.250546          20.808269              3.636080e-02   
         3         46.393078           0.126866              2.135912e-03   
         4         16.574735           0.026651              9.556637e-05   
         5         23.313601           0.162975             -2.835944e-06   
      4  1        221.890133           0.697629              1.260381e-03   
         2       2383.307694          12.355969              4.382082e-02   
         3         27.141401           0.074380              2.257220e-03   
         4         12.753129           0.020550              9.626123e-05   
         5          0.029416           0.000206             -5.221756e-08   
      5  1         60.534915           0.191555              6.182545e-05   
         2       2640.876022          13.779907              6.177238e-03   
         3         24.738421           0.068233              1.053543e-03   
         4         43.229507           0.070111              1.663721e-04   
         5          0.613044           0.004322              2.142812e-07   
      6  1         74.317492           0.237215              5.031547e-05   
         2        884.945801           4.657777             -9.721577e-04   
         3          9.622232           0.026771              5.031663e-04   
         4        125.341453           0.205052              5.015568e-04   
         5        322.703889           2.295151              8.137463e-05   
      7  1        153.934714           0.497847              1.619247e-03   
         2        404.015491           2.154605              7.671249e-03   
         3         15.725692           0.044331              4.229313e-04   
         4        378.043885           0.626642              7.001310e-04   
         5        378.292024           2.726099              1.799059e-05   
      8  1        644.331429           2.093089              1.571222e-03   
         2        204.697357           1.096481              1.554743e-03   
         3         18.746172           0.053080              2.727418e-04   
         4        362.119084           0.602904              3.402498e-04   
         5        427.243675           3.092498              1.296071e-04   
      9  1       1257.887651           4.095276              2.040587e-04   
         2        267.004802           1.433411              1.650232e-04   
         3         58.996584           0.167419              1.166029e-04   
         4        141.167951           0.235557              4.211397e-05   
         5        165.923680           1.203663              5.861686e-05   
      10 1          4.907521           0.015529              2.063902e-04   
         2          1.112647           0.005806              9.111089e-05   
         3          3.292439           0.009081              8.432018e-05   
         4         37.518037           0.060848              7.455402e-05   
         5         80.955615           0.570808              1.559794e-04   

          